# Module 1: Six-Pillar DAMA Data Hygiene & Structural Integrity Audit
**SQL Reference:** [`sql/01_data_audit.sql`](../sql/01_data_audit.sql) | **Visual Chart:** [`figures/01_data_hygiene_and_bounce.png`](../figures/01_data_hygiene_and_bounce.png)

## Executive Summary
Before computing conversion or revenue metrics, we audit `homework-data2020.saily_data_analyst.sessions` (`274,695` raw rows) and `homework-data2020.saily_data_analyst.events` (`295,041` raw rows) across the six DAMA dimensions:
1. **Uniqueness & Multi-Order Grain:** `unique_session_id` is **not** a primary key in `sessions` (`274,695` raw rows vs. **`241,548` distinct sessions**; `30,836` session IDs appear $\ge 2$ times, generating `33,147` surplus rows). Crucially, `transaction_id` (`25,133` valid orders totaling **`$358,625.85`**) is 100% unique (`0` duplicates) across **`24,456` distinct converting sessions** (`677` multi-plan orders within the same session).
2. **Three Conversion Rate Definitions:**
   - **Naive Unadjusted Row Ratio:** `25,133 / 274,695` = **`9.15%`** (flawed: mixes row denominator with order numerator).
   - **Strict Deduplicated Session Conversion Rate (Primary KPI):** `24,456 / 241,548` = **`10.12%`** (`+0.97 pp` / `+10.6%` relative uplift over naive).
   - **Order-to-Session Ratio (Transactions per Session):** `25,133 / 241,548` = **`10.40%`** (`1.028` orders per converting session).
3. **Referential Integrity & Deduplicated Bounce Rates:** `125,553` unique sessions (`51.98%`) record zero events (`70.78%` bounce on Android Incompatible vs. `47.53%` on Android Compatible and `50.98%` on iOS). Only `1` orphan session exists in `events` (`0.0004%`).


In [1]:
import pandas as pd
import numpy as np

# Verified BigQuery Output from sql/01_data_audit.sql (Query 1.1 & 1.2)
audit_summary = pd.DataFrame([
    {"Metric": "Total Raw Rows (sessions)", "Count": 274695, "Share_or_Rate": "100.00%", "Notes": "Raw table scan"},
    {"Metric": "Distinct Unique Sessions", "Count": 241548, "Share_or_Rate": "87.93% of rows", "Notes": "30,836 duplicated IDs (33,147 surplus rows)"},
    {"Metric": "Total Valid Paid Orders (transaction_id)", "Count": 25133, "Share_or_Rate": "$358,625.85", "Notes": "100% unique transaction_id ($14.27 AOV)"},
    {"Metric": "Distinct Converting Sessions (Buyers)", "Count": 24456, "Share_or_Rate": "10.12% Strict CR", "Notes": "677 multi-plan orders within same session"},
    {"Metric": "Naive Unadjusted CR (Orders / Raw Rows)", "Count": 25133, "Share_or_Rate": "9.15%", "Notes": "Distorted by 33,147 duplicate non-buyer rows"},
    {"Metric": "Order-to-Session Ratio (Orders / Sessions)", "Count": 25133, "Share_or_Rate": "10.40%", "Notes": "1.028 orders per converting session ($14.66 ARPPU)"}
])
print(audit_summary.to_string(index=False))


Metric  Count  Share_or_Rate                                         Notes
                 Total Raw Rows (sessions) 274695        100.00%                                Raw table scan
                  Distinct Unique Sessions 241548 87.93% of rows   30,836 duplicated IDs (33,147 surplus rows)
  Total Valid Paid Orders (transaction_id)  25133    $358,625.85       100% unique transaction_id ($14.27 AOV)
     Distinct Converting Sessions (Buyers)  24456 10.12% Strict CR     677 multi-plan orders within same session
   Naive Unadjusted CR (Orders / Raw Rows)  25133          9.15%  Distorted by 33,147 duplicate non-buyer rows
Order-to-Session Ratio (Orders / Sessions)  25133         10.40% 1.028 orders per converting session ($14.66 ARPPU)


In [2]:
# Verified BigQuery Output from sql/01_data_audit.sql (Query 1.3 & 1.5): Deduplicated Cohort Hygiene & Bounce Rate
cohort_hygiene = pd.DataFrame([
    {"Cohort": "iOS (not_applicable)", "Raw_Rows": 146359, "Unique_Sessions": 128872, "Zero_Event_Bounces": 65701, "Engaged_Sessions": 63171, "Bounce_Rate_Pct": 50.98, "Converting_Sessions": 16088, "Paid_Orders": 16527, "Revenue_USD": 240442.20},
    {"Cohort": "Android Compatible (true)", "Raw_Rows": 96827, "Unique_Sessions": 85603, "Zero_Event_Bounces": 40690, "Engaged_Sessions": 44913, "Bounce_Rate_Pct": 47.53, "Converting_Sessions": 8258, "Paid_Orders": 8495, "Revenue_USD": 116945.46},
    {"Cohort": "Android Incompatible (false)", "Raw_Rows": 31509, "Unique_Sessions": 27073, "Zero_Event_Bounces": 19162, "Engaged_Sessions": 7911, "Bounce_Rate_Pct": 70.78, "Converting_Sessions": 110, "Paid_Orders": 111, "Revenue_USD": 1238.19}
])
cohort_hygiene["Strict_Session_CR_Pct"] = (cohort_hygiene["Converting_Sessions"] / cohort_hygiene["Unique_Sessions"] * 100).round(2)
cohort_hygiene["AOV_USD"] = (cohort_hygiene["Revenue_USD"] / cohort_hygiene["Paid_Orders"]).round(2)
cohort_hygiene["ARPPU_USD"] = (cohort_hygiene["Revenue_USD"] / cohort_hygiene["Converting_Sessions"]).round(2)
print(cohort_hygiene.to_string(index=False))


Cohort  Raw_Rows  Unique_Sessions  Zero_Event_Bounces  Engaged_Sessions  Bounce_Rate_Pct  Converting_Sessions  Paid_Orders  Revenue_USD  Strict_Session_CR_Pct  AOV_USD  ARPPU_USD
        iOS (not_applicable)    146359           128872               65701             63171            50.98                16088        16527    240442.20                  12.48    14.55      14.95
   Android Compatible (true)     96827            85603               40690             44913            47.53                 8258         8495    116945.46                   9.65    13.77      14.16
Android Incompatible (false)     31509            27073               19162              7911            70.78                  110          111      1238.19                   0.41    11.15      11.26


## Key Governance Rules Established for Downstream Modules
1. **Session Deduplication Pattern:** Always collapse `sessions` to `1` row per `unique_session_id` before joining to `events` when computing funnel progression, conversion rates, or bounce rates.
2. **Revenue Preservation Pattern:** Aggregate `COUNT(transaction_id)` and `SUM(total_billings_in_usd)` at the `unique_session_id` grain prior to joining, preserving all `25,133` orders and `$358,625.85` in gross billings without Cartesian row inflation.
3. **Client-Side Loop Telemetry Guardrail:** `250` sessions (`0.10%`) fired an identical event `>15` times (up to `167` repeats); stage gates must be evaluated on `COUNT(DISTINCT unique_session_id)` rather than raw event counts.
